# 05 — Dataset supervisado  (auto-generado por build_notebook_05.py)

Construye el dataset supervisado para los corredores **E2** y **E59**:
ventanas deslizantes, normalización z-score por dirección, máscaras de
cardinalidad variable y un `HeadwayDataset` compatible con PyTorch DataLoader.

Referencia: `docs/plan-de-desarrollo.md §3 Fase 3 — Dataset supervisado`.

In [ ]:

import polars as pl
import numpy as np
from pathlib import Path

# Locate headways parquets for E2 and E59 under /kaggle/input or local dir.
def _find_parquet(empresa_id: int) -> Path:
    name = f"headways_E{empresa_id}.parquet"
    if Path("/kaggle/input").exists():
        candidates = list(Path("/kaggle/input").rglob(name))
        if candidates:
            return candidates[0]
    candidates = list(Path(".").rglob(name))
    if candidates:
        return candidates[0]
    raise FileNotFoundError(
        f"{name} not found. Expected at /kaggle/input/**/{name}"
    )

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUTPUT_DIR.mkdir(exist_ok=True)
STATS_CSV = OUTPUT_DIR / "dataset_stats.csv"

print(f"Output dir: {OUTPUT_DIR}")

## Module: evaluation/splits

Temporal split helper (`split_temporal`) and train-only p99 winsorization
(`winsorize_train_p99`).  Split date ranges are locked in spec §3.

In [ ]:
"""Temporal split and winsorization helpers for headway evaluation — Fase 3.

Public API:
    split_temporal(df: pl.DataFrame) -> pl.DataFrame
    winsorize_train_p99(df: pl.DataFrame) -> tuple[pl.DataFrame, float]

Constants (split date ranges, locked in spec §3 and design §5):
    SPLIT_TRAIN_START, SPLIT_TRAIN_END
    SPLIT_VAL_START,   SPLIT_VAL_END
    SPLIT_TEST_START,  SPLIT_TEST_END
    WINSOR_QUANTILE

Design decisions (locked in design §5 and §9):
  - Split key is pl.col("t").dt.date() membership, NOT row index.
  - Three ranges are exhaustive and mutually exclusive.
  - Rows outside all three ranges receive None (split column = null).
  - Winsorization threshold is computed on train rows only (AC-WINSOR-1, AC-WINSOR-2).
  - Null delta_t_min rows are NOT clipped (AC-WINSOR-3).
  - Rows above threshold are clipped (not dropped) (AC-WINSOR-4).
  - Constants live here (not PRODUCTIVE_PARAMS) — evaluation protocol concern.
  - WINSOR_QUANTILE and split dates are not added to pyproject.toml.
"""
from __future__ import annotations

from datetime import date

import polars as pl

# ---------------------------------------------------------------------------
# Split date range constants (spec §3, inclusive on both ends)
# ---------------------------------------------------------------------------

SPLIT_TRAIN_START: date = date(2023, 10, 1)
SPLIT_TRAIN_END:   date = date(2024, 1, 15)

SPLIT_VAL_START:   date = date(2024, 1, 16)
SPLIT_VAL_END:     date = date(2024, 2, 7)

SPLIT_TEST_START:  date = date(2024, 2, 8)
SPLIT_TEST_END:    date = date(2024, 2, 29)

WINSOR_QUANTILE: float = 0.99


def split_temporal(df: pl.DataFrame) -> pl.DataFrame:
    """Add a `split` column (Utf8) with values {"train", "val", "test"}.

    Membership is determined by pl.col("t").dt.date() against the six
    module-level date constants.  Rows outside all three ranges receive
    null (should not exist in the R7 v4 dataset; harness raises if found).

    Parameters
    ----------
    df:
        headways DataFrame containing at least a `t` (Datetime) column.

    Returns
    -------
    pl.DataFrame — input frame with one added column `split: Utf8`.
    """
    day = pl.col("t").dt.date()
    return df.with_columns(
        pl.when((day >= SPLIT_TRAIN_START) & (day <= SPLIT_TRAIN_END))
          .then(pl.lit("train"))
          .when((day >= SPLIT_VAL_START) & (day <= SPLIT_VAL_END))
          .then(pl.lit("val"))
          .when((day >= SPLIT_TEST_START) & (day <= SPLIT_TEST_END))
          .then(pl.lit("test"))
          .otherwise(None)
          .alias("split")
    )


def winsorize_train_p99(
    df: pl.DataFrame,
) -> tuple[pl.DataFrame, float]:
    """Clip delta_t_min to the 99th-percentile threshold computed on train rows only.

    The threshold is computed once as a scalar from non-null train-split rows.
    It is then applied as a clip ceiling to ALL rows (train + val + test).
    Null delta_t_min values are never clipped — they remain null (AC-WINSOR-3).

    Parameters
    ----------
    df:
        headways DataFrame that already has a `split` column (added by
        split_temporal) and a `delta_t_min` (Float64 nullable) column.

    Returns
    -------
    (clipped_df, threshold)
        clipped_df: same schema as df, delta_t_min clipped.
        threshold: the scalar train-p99 value used as the clip ceiling.

    Design note (AC-WINSOR-2 leakage guard):
        The filter `split == "train"` is applied BEFORE computing the quantile,
        so extreme outliers in val or test rows cannot shift the threshold.
    """
    threshold = float(
        df.filter(
            (pl.col("split") == "train") & pl.col("delta_t_min").is_not_null()
        )["delta_t_min"]
        .quantile(WINSOR_QUANTILE)
    )

    # Clip: preserve null rows; clip non-null rows to threshold from above.
    # pl.min_horizontal(col, lit(threshold)) would coerce null → 0 in some
    # polars versions, so we use the explicit when/then pattern (design §5).
    clipped = df.with_columns(
        pl.when(pl.col("delta_t_min").is_null())
          .then(None)
          .otherwise(
              pl.min_horizontal(pl.col("delta_t_min"), pl.lit(threshold))
          )
          .alias("delta_t_min")
    )
    return clipped, threshold

## Module: evaluation/metrics

`mae` and `rmse` in minutes.  Included for downstream sanity checks.

In [ ]:
"""Evaluation metrics for headway forecasting — Fase 3.

Public API:
    mae(y_true, y_pred) -> float
    rmse(y_true, y_pred) -> float

Both functions accept polars Series (Float64) or numpy arrays (float64).
Null / NaN masking: rows where EITHER y_true or y_pred is null/NaN are
dropped before aggregation.  If no valid rows remain, ValueError is raised.

Design decisions locked in design §4:
  - ValueError on empty/all-null input (NOT silent NaN return).
  - Only MAE and RMSE are in scope (spec B3-NO-MAPE — ratio-based metrics
    are out of scope because near-zero headways cause denominator blow-up).
  - No new pyproject.toml dependencies (polars + numpy already present).
"""
from __future__ import annotations

import numpy as np
import polars as pl


def _to_numpy_with_mask(
    y_true: pl.Series | np.ndarray,
    y_pred: pl.Series | np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Coerce both inputs to float64 numpy arrays and apply the null/NaN mask.

    Polars Series with dtype Float64: null cells become NaN via .to_numpy().
    numpy arrays: assumed to already use NaN for missing values.

    Returns
    -------
    (y_true_masked, y_pred_masked) — two 1-D float64 arrays of equal length,
    containing no NaN values.  May be empty if all rows were masked.
    """
    # Coerce to numpy.
    if isinstance(y_true, pl.Series):
        yt = y_true.to_numpy(allow_copy=True).astype(np.float64)
    else:
        yt = np.asarray(y_true, dtype=np.float64).ravel()

    if isinstance(y_pred, pl.Series):
        yp = y_pred.to_numpy(allow_copy=True).astype(np.float64)
    else:
        yp = np.asarray(y_pred, dtype=np.float64).ravel()

    # Elementwise mask: keep row only if BOTH sides are finite (not NaN).
    mask = ~(np.isnan(yt) | np.isnan(yp))
    return yt[mask], yp[mask]


def mae(
    y_true: pl.Series | np.ndarray,
    y_pred: pl.Series | np.ndarray,
) -> float:
    """Mean Absolute Error in minutes, with null/NaN masking.

    Parameters
    ----------
    y_true, y_pred:
        Ground-truth and predicted headway values in minutes.
        Accepts polars Series (Float64) or numpy arrays (float64).
        Null / NaN positions in either input are dropped before computation.

    Returns
    -------
    float — MAE in minutes.

    Raises
    ------
    ValueError
        If the masked input is empty (all-null or zero-length).
    """
    yt, yp = _to_numpy_with_mask(y_true, y_pred)
    if len(yt) == 0:
        raise ValueError(
            "mae: metric on empty/all-null input — no valid (y_true, y_pred) pairs."
        )
    return float(np.mean(np.abs(yt - yp)))


def rmse(
    y_true: pl.Series | np.ndarray,
    y_pred: pl.Series | np.ndarray,
) -> float:
    """Root Mean Squared Error in minutes, with null/NaN masking.

    Parameters
    ----------
    y_true, y_pred:
        Ground-truth and predicted headway values in minutes.
        Accepts polars Series (Float64) or numpy arrays (float64).
        Null / NaN positions in either input are dropped before computation.

    Returns
    -------
    float — RMSE in minutes.

    Raises
    ------
    ValueError
        If the masked input is empty (all-null or zero-length).
    """
    yt, yp = _to_numpy_with_mask(y_true, y_pred)
    if len(yt) == 0:
        raise ValueError(
            "rmse: metric on empty/all-null input — no valid (y_true, y_pred) pairs."
        )
    return float(np.sqrt(np.mean((yt - yp) ** 2)))

## Module: data/windowing

`make_window_index` — per-slot deterministic window index.
`compute_max_N` — train-p99 of (n_buses - 1) per (empresaid, direction).
Constants: `DEFAULT_T_IN=12`, `DEFAULT_T_OUT=1`, `DEFAULT_STRIDE=1`.

In [ ]:
"""Windowing module for supervised dataset construction — Fase 3 DL.

AC-WIN-1: Build window index per slot.
AC-WIN-2: Stride-parametrized index generation.
AC-WIN-3: Deterministic slot-boundary-respecting index.
AC-WIN-4: Exported constants DEFAULT_T_IN, DEFAULT_T_OUT, DEFAULT_STRIDE.
AC-WIN-5: Empty-slot guard (returns zero entries when N < T_in + T_out).
AC-WIN-6: Zero torch imports at module level.
AC-MAXN-1: compute_max_N returns train-p99 of (n_buses-1) per (empresaid, direction).
AC-MAXN-2: compute_max_N is called on train-only df; leakage is caller responsibility.

Design decisions (locked in design §2.2 and §5):
  - WindowIndexEntry: TypedDict with empresaid, direction, pair_rank, start_idx.
  - start_idx is relative to the sorted slot frame (not the full df).
  - Slot key: (empresaid, direction, pair_rank).
  - No torch imports anywhere in this module (INV-10, DL-10).
"""
from __future__ import annotations

import math
from typing import TypedDict

import polars as pl

# ---------------------------------------------------------------------------
# Constants (locked in design §5 — DL-1)
# ---------------------------------------------------------------------------

DEFAULT_T_IN: int = 12
DEFAULT_T_OUT: int = 1
DEFAULT_STRIDE: int = 1

_SLOT_COLS: list[str] = ["empresaid", "direction", "pair_rank"]


class WindowIndexEntry(TypedDict):
    """Single window anchor.

    empresaid: int — corridor identifier.
    direction: int — bus direction (-1 or +1).
    pair_rank: int — positional slot index within a snapshot.
    start_idx: int — row index into the sorted slot frame where this window starts.
                     The window covers rows [start_idx, start_idx + T_in + T_out).
    """

    empresaid: int
    direction: int
    pair_rank: int
    start_idx: int


# ---------------------------------------------------------------------------
# Internal helpers
# ---------------------------------------------------------------------------

def _slot_lengths(df: pl.DataFrame) -> pl.DataFrame:
    """Return a DataFrame with (empresaid, direction, pair_rank, n_rows).

    Used by make_window_index to determine how many windows each slot produces.
    The count is over ALL rows (null delta_t_min counts — windowing does not
    drop null rows; the Dataset layer handles null masking later).
    """
    return (
        df.group_by(_SLOT_COLS)
        .agg(pl.len().alias("n_rows"))
    )


# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------

def compute_max_N(
    train_df: pl.DataFrame,
    *,
    quantile: float = 0.99,
) -> dict[tuple[int, int], int]:
    """Train-p99 of (n_buses - 1) per (empresaid, direction). DL-5. AC-MAXN-1..2.

    Parameters
    ----------
    train_df:
        DataFrame filtered to train rows only (caller responsibility).
        Must have columns: empresaid (Int64), direction (Int64), n_buses (Int32).
    quantile:
        Percentile for the cap (default 0.99 per DL-5).

    Returns
    -------
    dict[(empresaid, direction), int] — the maximum slot index (0-based max_N).
    Returned values are Python int (not np.int64) so they can be used as tensor
    dimensions directly.
    """
    # Compute quantile of (n_buses - 1) per (empresaid, direction).
    # We use unique snapshots: each row in the windowing context represents one
    # (empresaid, direction, snapshot) combination. n_buses is per snapshot.
    result: dict[tuple[int, int], int] = {}

    # Group by (empresaid, direction) and compute the p99 of (n_buses - 1).
    stats = (
        train_df
        .with_columns(
            (pl.col("n_buses") - 1).alias("_n_slots")
        )
        .group_by(["empresaid", "direction"])
        .agg(
            pl.col("_n_slots").quantile(quantile).alias("max_N_float")
        )
    )

    for row in stats.iter_rows(named=True):
        key = (int(row["empresaid"]), int(row["direction"]))
        result[key] = int(math.floor(row["max_N_float"]))

    return result


def make_window_index(
    df: pl.DataFrame,
    *,
    T_in: int = DEFAULT_T_IN,
    T_out: int = DEFAULT_T_OUT,
    stride: int = DEFAULT_STRIDE,
) -> list[WindowIndexEntry]:
    """Deterministic per-slot window index. DL-1, DL-11. AC-WIN-1..5.

    Produces a list of WindowIndexEntry dicts where each entry anchors one
    training window. Entries are sorted by (empresaid, direction, pair_rank,
    start_idx) for determinism.

    Parameters
    ----------
    df:
        headways DataFrame sorted (or sortable) by (slot_cols, t).
        Columns required: empresaid, direction, pair_rank, t.
    T_in:
        Input sequence length (number of timesteps fed to model).
    T_out:
        Prediction horizon (number of future timesteps).
    stride:
        Step between consecutive window starts (default 1 = every timestep).

    Returns
    -------
    list[WindowIndexEntry] — may be empty if no slot has enough rows.
    """
    window_size = T_in + T_out
    index: list[WindowIndexEntry] = []

    # Partition by slot to keep slot boundaries clean (AC-WIN-3).
    slots = df.sort(_SLOT_COLS + ["t"]).partition_by(_SLOT_COLS, maintain_order=True)

    for slot_df in slots:
        if slot_df.is_empty():
            continue

        n_rows = len(slot_df)
        if n_rows < window_size:
            # AC-WIN-5: not enough rows for even one window — skip.
            continue

        # Extract slot key from first row.
        first = slot_df.row(0, named=True)
        emp: int = int(first["empresaid"])
        direction: int = int(first["direction"])
        pr: int = int(first["pair_rank"])

        # Generate start indices with stride.
        # Number of valid windows: floor((n_rows - window_size) / stride) + 1
        n_windows = math.floor((n_rows - window_size) / stride) + 1
        for w in range(n_windows):
            start_idx = w * stride
            index.append(
                WindowIndexEntry(
                    empresaid=emp,
                    direction=direction,
                    pair_rank=pr,
                    start_idx=start_idx,
                )
            )

    return index

## Module: data/normalization

`compute_normalization_stats` — per-direction z-score stats from TRAIN ONLY.
`apply_zscore` — add `delta_t_min_z` column; no clipping (DL-8).

In [ ]:
"""Normalization module for supervised dataset construction — Fase 3 DL.

AC-NORM-1: compute_normalization_stats uses TRAIN ROWS ONLY.
AC-NORM-2: apply_zscore = (x - mean) / (std + Z_EPS) per (empresaid, direction).
AC-NORM-3: null delta_t_min passes through as null in the output column.
AC-NORM-4: no clipping — values with |z| > 5 are passed through unmodified (DL-8).
AC-NORM-5: zero torch imports at module level (INV-10).
AC-LEAK-1: leakage guard — caller must pass train_df only; this module does not filter.

Pre-condition: input df must already be winsorized via winsorize_train_p99 (INV-6).
Design decisions locked in design §2.3 and §5.
"""
from __future__ import annotations

from dataclasses import dataclass

import polars as pl

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

Z_EPS: float = 1e-8  # numerical safety: (x - mean) / (std + Z_EPS)


# ---------------------------------------------------------------------------
# Data types
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class NormalizationStats:
    """Per-(empresaid, direction) z-score parameters. Pure data, no torch.

    Attributes
    ----------
    means:
        Mean of delta_t_min per (empresaid, direction) computed from train rows only.
    stds:
        Standard deviation of delta_t_min per (empresaid, direction) from train only.
    """

    means: dict[tuple[int, int], float]
    stds: dict[tuple[int, int], float]


# ---------------------------------------------------------------------------
# Internal helpers
# ---------------------------------------------------------------------------

def _lookup_expr(
    stats: NormalizationStats,
    kind: str,
) -> pl.Expr:
    """Build a polars conditional expression mapping (empresaid, direction) → mean|std.

    Instead of a Python row-by-row loop, we build a pl.when/then chain over all
    known (empresa, direction) keys. Unknown keys return 0.0 (should not happen
    in a correctly filtered input; callers are expected to pass frames that
    only contain corridors present in train).

    Parameters
    ----------
    stats:
        NormalizationStats holding all known keys.
    kind:
        Either "mean" or "std".
    """
    lookup = stats.means if kind == "mean" else stats.stds

    if not lookup:
        return pl.lit(0.0)

    items = list(lookup.items())
    (emp0, dir0), val0 = items[0]
    expr = pl.when(
        (pl.col("empresaid") == emp0) & (pl.col("direction") == dir0)
    ).then(pl.lit(val0))

    for (emp, direction), val in items[1:]:
        expr = expr.when(
            (pl.col("empresaid") == emp) & (pl.col("direction") == direction)
        ).then(pl.lit(val))

    return expr.otherwise(pl.lit(0.0))


# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------

def compute_normalization_stats(
    train_df: pl.DataFrame,
) -> NormalizationStats:
    """Mean/std of delta_t_min per (empresaid, direction) from TRAIN ROWS ONLY.

    AC-NORM-1: caller must pass a train-only DataFrame (no leakage protection
    inside this function — leakage guard is the caller's responsibility per INV-2).

    Null delta_t_min rows are excluded from the computation (standard mean/std
    ignores nulls in polars by default).

    Parameters
    ----------
    train_df:
        DataFrame with train rows only. Required columns: empresaid (Int64),
        direction (Int64), delta_t_min (Float64 nullable).

    Returns
    -------
    NormalizationStats with means and stds dicts keyed by (empresaid, direction).
    """
    agg = (
        train_df
        .group_by(["empresaid", "direction"])
        .agg(
            pl.col("delta_t_min").mean().alias("mean"),
            pl.col("delta_t_min").std().alias("std"),
        )
    )

    means: dict[tuple[int, int], float] = {}
    stds: dict[tuple[int, int], float] = {}

    for row in agg.iter_rows(named=True):
        key = (int(row["empresaid"]), int(row["direction"]))
        means[key] = float(row["mean"]) if row["mean"] is not None else 0.0
        stds[key] = float(row["std"]) if row["std"] is not None else 0.0

    return NormalizationStats(means=means, stds=stds)


def apply_zscore(
    df: pl.DataFrame,
    stats: NormalizationStats,
    *,
    out_col: str = "delta_t_min_z",
) -> pl.DataFrame:
    """Add z-scored column: (delta_t_min - mean) / (std + Z_EPS) per (empresa, direction).

    AC-NORM-2: formula is (x - mean) / (std + Z_EPS).
    AC-NORM-3: null delta_t_min rows produce null in out_col (no imputation).
    AC-NORM-4 + DL-8: no clipping — values with |z| > 5 pass through unchanged.

    Parameters
    ----------
    df:
        DataFrame to z-score. May be train, val, or test split. Required columns:
        empresaid, direction, delta_t_min.
    stats:
        NormalizationStats from compute_normalization_stats (train only).
    out_col:
        Name for the output z-scored column (default: delta_t_min_z).

    Returns
    -------
    pl.DataFrame — input frame with out_col (Float64 nullable) added.
    """
    mean_expr = _lookup_expr(stats, "mean")
    std_expr = _lookup_expr(stats, "std")

    return df.with_columns(
        pl.when(pl.col("delta_t_min").is_null())
        .then(None)
        .otherwise(
            (pl.col("delta_t_min") - mean_expr) / (std_expr + Z_EPS)
        )
        .alias(out_col)
        .cast(pl.Float64)
    )

## Module: data/context_features

`encode_context` — add 5 cyclical + atypical-flag columns.
`load_atypical_days` — graceful fallback to empty set when CSV absent (DL-2).

In [ ]:
"""Context features module for supervised dataset construction — Fase 3 DL.

AC-CTX-1: encode_context adds hour_sin, hour_cos at midnight → (0, 1).
AC-CTX-2: encode_context adds dow_sin, dow_cos with period 7; emits 5 named columns.
AC-CTX-3: load_atypical_days(None) returns empty set (graceful fallback, DL-2).
AC-CTX-4: load_atypical_days(path) returns set[date] from CSV when file exists.
AC-CTX-5: atypical_flag=1.0 when timestamp date in atypical_dates, else 0.0.
AC-CTX-6: zero torch imports at module level (INV-10, DL-10).

Design decisions locked in design §2.4 and §5:
  - encode_context operates on a DataFrame with a `t` (Datetime) column.
  - Cyclical encoding: sin(2π * value / period), cos(2π * value / period).
  - atypical_flag = 1.0 when t.date() in atypical_dates else 0.0.
  - DL-2: graceful fallback to atypical_flag=0 when path is None or missing.
  - No torch imports (INV-10).
"""
from __future__ import annotations

import logging
import math
import warnings
from datetime import date
from pathlib import Path

import polars as pl

_log = logging.getLogger(__name__)

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

CONTEXT_FEATURE_NAMES: tuple[str, ...] = (
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "atypical_flag",
)


# ---------------------------------------------------------------------------
# Internal helpers
# ---------------------------------------------------------------------------

def _cyclical_pair(col: pl.Expr, period: int, prefix: str) -> list[pl.Expr]:
    """Emit [sin_expr, cos_expr] aliased <prefix>_sin, <prefix>_cos.

    Encoding: sin(2π * col / period), cos(2π * col / period).

    Parameters
    ----------
    col:
        Polars expression that yields a numeric value (e.g. hour 0-23, dow 0-6).
    period:
        Full cycle length (24 for hour, 7 for day-of-week).
    prefix:
        Column name prefix ("hour" or "dow").
    """
    angle = col * (2.0 * math.pi / period)
    return [
        angle.sin().alias(f"{prefix}_sin"),
        angle.cos().alias(f"{prefix}_cos"),
    ]


# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------

def encode_context(
    df: pl.DataFrame,
    *,
    atypical_dates: set[date] | None = None,
) -> pl.DataFrame:
    """Add 5 context columns derived from the `t` (Datetime) column.

    AC-CTX-1..5. DL-2 graceful fallback: atypical_flag=0 when atypical_dates
    is None or empty.

    Parameters
    ----------
    df:
        DataFrame with a `t` (Datetime[us]) column.
    atypical_dates:
        Set of dates that are atypical (e.g. holidays, strikes). When None
        or empty, atypical_flag is 0.0 for all rows.

    Returns
    -------
    pl.DataFrame — input frame with 5 additional columns appended in the order
    defined by CONTEXT_FEATURE_NAMES.
    """
    if atypical_dates is None:
        atypical_dates = set()

    # Cyclical hour and day-of-week encodings.
    # polars dt.weekday() returns ISO weekday: Monday=1 .. Sunday=7.
    # We convert to 0-indexed (Monday=0 .. Sunday=6) to align with Python convention
    # so that midnight Monday → dow=0 → dow_sin=sin(0)=0, dow_cos=cos(0)=1 (AC-CTX-1).
    hour_expr = pl.col("t").dt.hour().cast(pl.Float64)
    dow_expr = (pl.col("t").dt.weekday() - 1).cast(pl.Float64)

    sin_cos_exprs: list[pl.Expr] = [
        *_cyclical_pair(hour_expr, 24, "hour"),
        *_cyclical_pair(dow_expr, 7, "dow"),
    ]

    # Atypical flag: 1.0 if the date is in the atypical set, else 0.0.
    if atypical_dates:
        # Build a list of date literals to check membership against.
        atypical_list = sorted(atypical_dates)
        date_col = pl.col("t").dt.date()
        flag_expr = pl.lit(0.0)

        # Chain when/then for each atypical date.
        flag_chain = pl.when(
            date_col == pl.lit(atypical_list[0])
        ).then(pl.lit(1.0))
        for d in atypical_list[1:]:
            flag_chain = flag_chain.when(
                date_col == pl.lit(d)
            ).then(pl.lit(1.0))
        flag_expr = flag_chain.otherwise(pl.lit(0.0))
    else:
        flag_expr = pl.lit(0.0)

    return df.with_columns(
        *sin_cos_exprs,
        flag_expr.cast(pl.Float64).alias("atypical_flag"),
    )


def load_atypical_days(
    path: Path | str | None,
) -> set[date]:
    """Read CSV with at least a `date` column; return set[date].

    AC-CTX-3 + DL-2: returns empty set when path is None OR file does not exist.
    A warning is emitted when the path is non-None but missing (so callers know
    the fallback was triggered — not a silent failure).

    Parameters
    ----------
    path:
        Path to a CSV file with a `date` column (ISO-8601 format).
        May be None, a string, or a Path object.

    Returns
    -------
    set[date] — parsed dates, or empty set on fallback.
    """
    if path is None:
        return set()

    resolved = Path(path)
    if not resolved.exists():
        warnings.warn(
            f"load_atypical_days: file not found at '{resolved}'; "
            "falling back to empty atypical set (atypical_flag=0 for all rows). "
            "DL-2 graceful fallback.",
            stacklevel=2,
        )
        return set()

    df = pl.read_csv(resolved, try_parse_dates=True)
    if "date" not in df.columns:
        warnings.warn(
            f"load_atypical_days: CSV at '{resolved}' has no 'date' column; "
            "falling back to empty set.",
            stacklevel=2,
        )
        return set()

    dates: set[date] = set()
    for val in df["date"].to_list():
        if val is not None:
            if isinstance(val, date):
                dates.add(val)
            else:
                try:
                    from datetime import datetime as _dt
                    dates.add(_dt.fromisoformat(str(val)).date())
                except ValueError:
                    _log.warning("Skipping unparseable date value: %s", val)

    return dates

## Module: data/dataset  (first torch import)

`HeadwayDataset` — on-the-fly window materialization with masks (DL-11).
`collate_fn` — batch stacking for variable-N edge cases (REQ-6).

In [ ]:
"""HeadwayDataset — torch adapter for supervised dataset construction.

This is the ONLY module in src/data/ that imports torch (INV-10, DL-10).
All other modules (windowing, normalization, context_features) are torch-free.

ACs covered:
    AC-DS-1: __getitem__ returns dict with keys {input, target, input_mask, target_mask, context}.
    AC-DS-2: tensor shapes per item: input (T_in, max_N), target (T_out, max_N),
             masks same as data, context (T_in, 5).
    AC-DS-3: float32 for input/target/context; bool for masks.
    AC-DS-4: len(dataset) == len(window_index).
    AC-DS-5: collate_fn stacks dicts into batched tensors on dim 0.
    AC-DS-6: DataLoader(dataset, collate_fn=collate_fn) iterates without error.
    AC-MASK-1: present non-null slot → mask True (True = VALID, INV-5).
    AC-MASK-2: absent slot → mask False, value 0.0.
    AC-MASK-3: present but null delta_t_min → mask False, value 0.0.
    AC-MASK-4: mask convention identical between input_mask and target_mask.
    AC-DS-NOMAT-1 / INV-7: __init__ MUST NOT call __getitem__ or iterate windows.

Design refs: spec §4 (AC-DS-*, AC-MASK-*), design §2.5, §4, §5, INV-4, INV-5, INV-7.
"""
from __future__ import annotations

from typing import Any

import polars as pl
import torch
from torch.utils.data import Dataset


# Re-export CONTEXT_FEATURE_NAMES for __init__.py convenience.

_SLOT_COLS: list[str] = ["empresaid", "direction", "pair_rank"]


class HeadwayDataset(Dataset):
    """Snapshot-as-set dataset backed by a precomputed window index.

    Returns one dict of 5 tensors per __getitem__ call; windows are
    materialized on-the-fly (never at __init__ time per INV-7 / DL-11).

    Tensor contract (per item, before DataLoader batching):
        input       : (T_in, max_N)   float32   — z-scored delta_t_min; 0 where absent
        target      : (T_out, max_N)  float32   — same for the horizon
        input_mask  : (T_in, max_N)   bool      — True where slot present AND non-null
        target_mask : (T_out, max_N)  bool      — same convention for horizon
        context     : (T_in, 5)       float32   — cyclical time + atypical flag

    Mask polarity: True = VALID (PyTorch attention_mask convention, INV-5).
    """

    def __init__(
        self,
        df: pl.DataFrame,
        window_index: list[WindowIndexEntry],
        *,
        max_N_by_direction: dict[tuple[int, int], int],
        T_in: int,
        T_out: int,
        value_col: str = "delta_t_min_z",
        context_cols: tuple[str, ...] = CONTEXT_FEATURE_NAMES,
    ) -> None:
        """Construct lightweight wrapper. AC-DS-NOMAT-1: MUST NOT iterate windows.

        Parameters
        ----------
        df:
            Full headways DataFrame (any split) that has already been:
            winsorized, z-scored (delta_t_min_z column present), and
            context-feature encoded (CONTEXT_FEATURE_NAMES columns present).
        window_index:
            Precomputed list of WindowIndexEntry dicts from make_window_index.
        max_N_by_direction:
            Per-(empresaid, direction) maximum slot count (0-indexed, train-p99).
        T_in:
            Number of input timesteps per window.
        T_out:
            Number of target timesteps per window.
        value_col:
            Name of the z-scored column in df (default: "delta_t_min_z").
        context_cols:
            Ordered tuple of context column names (must be 5 columns, float64).
        """
        # Store metadata — NO window materialization here (INV-7).
        self._df = df
        self._window_index = window_index
        self._max_N_by_direction = max_N_by_direction
        self._T_in = T_in
        self._T_out = T_out
        self._value_col = value_col
        self._context_cols = list(context_cols)

        # Cache per-slot partitions lazily (populated on first access per slot).
        # Key: (empresaid, direction, pair_rank) → sorted slot DataFrame.
        self._slot_cache: dict[tuple[int, int, int], pl.DataFrame] = {}

    # ------------------------------------------------------------------
    # Dataset protocol
    # ------------------------------------------------------------------

    def __len__(self) -> int:
        """AC-DS-4: total number of windows across all slots."""
        return len(self._window_index)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        """Materialize one window. AC-DS-1, AC-DS-2, AC-DS-3.

        Returns
        -------
        dict with keys: input, target, input_mask, target_mask, context.
        """
        entry = self._window_index[idx]
        empresaid: int = entry["empresaid"]
        direction: int = entry["direction"]
        pair_rank: int = entry["pair_rank"]
        start_idx: int = entry["start_idx"]

        return self._materialize_window(
            empresaid=empresaid,
            direction=direction,
            pair_rank=pair_rank,
            start_idx=start_idx,
        )

    # ------------------------------------------------------------------
    # Internal helpers
    # ------------------------------------------------------------------

    def _slot_frame(self, empresaid: int, direction: int, pair_rank: int) -> pl.DataFrame:
        """Return the sorted slot frame, cached per slot key.

        The frame is sorted by 't' once and reused across all windows
        that share the same slot.
        """
        key = (empresaid, direction, pair_rank)
        if key not in self._slot_cache:
            slot_df = (
                self._df
                .filter(
                    (pl.col("empresaid") == empresaid)
                    & (pl.col("direction") == direction)
                    & (pl.col("pair_rank") == pair_rank)
                )
                .sort("t")
            )
            self._slot_cache[key] = slot_df
        return self._slot_cache[key]

    def _materialize_window(
        self,
        *,
        empresaid: int,
        direction: int,
        pair_rank: int,
        start_idx: int,
    ) -> dict[str, torch.Tensor]:
        """Build input/target/mask/context tensors for one window.

        Design note: the window_index records start_idx relative to the sorted
        slot frame for (empresaid, direction, pair_rank). We slice T_in rows for
        the input and T_out rows for the target; then pad the N dimension
        to max_N using per-snapshot presence detection.
        """
        window_size = self._T_in + self._T_out
        max_N = self._max_N_by_direction[(empresaid, direction)]

        # Get the sorted slot frame.
        slot_df = self._slot_frame(empresaid, direction, pair_rank)

        # Slice the window rows.
        window_df = slot_df.slice(start_idx, window_size)

        # Extract value column and context columns.
        # The slot frame holds ONE pair_rank at a time; we need all pair_ranks
        # for this snapshot to build the full (T, max_N) tensors.
        # We use the snapshot timestamps from this slot to locate all pair_ranks.
        timestamps = window_df["t"].to_list()

        # For each snapshot timestep, gather all pair_ranks in [0, max_N).
        # This requires a lookup in the full df filtered to (empresaid, direction).
        # We cache the direction-level frame for efficiency.
        dir_key = (empresaid, direction)
        if not hasattr(self, "_dir_cache"):
            self._dir_cache: dict[tuple[int, int], pl.DataFrame] = {}
        if dir_key not in self._dir_cache:
            self._dir_cache[dir_key] = (
                self._df
                .filter(
                    (pl.col("empresaid") == empresaid)
                    & (pl.col("direction") == direction)
                )
            )
        dir_df = self._dir_cache[dir_key]

        # Build tensors by iterating over the window timesteps.
        # We build dense (T, max_N) matrices where absent pair_ranks are zero / False.
        T = window_size

        # Pre-allocate: float tensors default 0.0, bool mask default False.
        values = torch.zeros((T, max_N), dtype=torch.float32)
        masks = torch.zeros((T, max_N), dtype=torch.bool)
        context = torch.zeros((T, len(self._context_cols)), dtype=torch.float32)

        for t_idx, ts in enumerate(timestamps):
            # Filter dir_df to this exact snapshot timestamp.
            snap = dir_df.filter(pl.col("t") == ts)

            # Context is the same per snapshot across pair_ranks — take first row.
            if not snap.is_empty():
                ctx_row = snap.row(0, named=True)
                for c_idx, col_name in enumerate(self._context_cols):
                    if col_name in ctx_row and ctx_row[col_name] is not None:
                        context[t_idx, c_idx] = float(ctx_row[col_name])

            # Fill values and masks per pair_rank present in this snapshot.
            for pr in snap["pair_rank"].to_list():
                if pr < 0 or pr >= max_N:
                    # Truncate pair_ranks beyond max_N (AC-MAXN-2).
                    continue
                pr_row = snap.filter(pl.col("pair_rank") == pr)
                if pr_row.is_empty():
                    continue
                val = pr_row[self._value_col][0]
                if val is not None:
                    values[t_idx, pr] = float(val)
                    masks[t_idx, pr] = True
                # If val is None: value stays 0.0, mask stays False (AC-MASK-3).

        return {
            "input": values[: self._T_in],
            "target": values[self._T_in :],
            "input_mask": masks[: self._T_in],
            "target_mask": masks[self._T_in :],
            "context": context[: self._T_in],
        }


# ---------------------------------------------------------------------------
# collate_fn
# ---------------------------------------------------------------------------

def collate_fn(
    batch: list[dict[str, torch.Tensor]],
) -> dict[str, torch.Tensor]:
    """Stack a list of __getitem__ outputs into batched tensors (B-axis prepended).

    All tensors in a batch share the same shape (T, max_N or 5) since max_N is
    fixed per (empresaid, direction) and all items in a batch should come from
    the same direction. torch.stack is used (not pad_sequence) because shapes
    are guaranteed equal.

    AC-DS-5: batch dimension is dim 0 for all tensors.
    AC-DS-6: compatible with torch.utils.data.DataLoader.
    """
    keys = list(batch[0].keys())
    return {k: torch.stack([item[k] for item in batch], dim=0) for k in keys}

## Cargar datos — E2 y E59

Lee los parquets v8 generados por el notebook 04 (NB04b, kernel_sources:
`alexhuaracha/04-preprocessing`).  Inyecta `empresaid` como columna literal.

Row-count assertion vs dataset-manifest.md v8 pins (AC-NB-5, DL-7):
  - E2:  1,009,284 rows
  - E59: 2,069,193 rows

In [ ]:

# empresaid is implicit in the filename in the v8 parquets — inject it as a
# literal column so it matches the supervised-dataset contract (slot key requires it).
hw_e2  = pl.read_parquet(_find_parquet(2)).with_columns(pl.lit(2,  dtype=pl.Int64).alias("empresaid"))
hw_e59 = pl.read_parquet(_find_parquet(59)).with_columns(pl.lit(59, dtype=pl.Int64).alias("empresaid"))

# Row-count assertion against dataset-manifest.md v8 values (AC-NB-5, R-KERNEL-SOURCES-PIN).
E2_EXPECTED_ROWS  = 1_009_284
E59_EXPECTED_ROWS = 2_069_193
assert hw_e2.height == E2_EXPECTED_ROWS, (
    f"E2 row count mismatch: expected {E2_EXPECTED_ROWS:,}, got {hw_e2.height:,}. "
    "Is kernel_sources pointing to the correct NB04 run?"
)
assert hw_e59.height == E59_EXPECTED_ROWS, (
    f"E59 row count mismatch: expected {E59_EXPECTED_ROWS:,}, got {hw_e59.height:,}. "
    "Is kernel_sources pointing to the correct NB04 run?"
)

print(f"E2:  {hw_e2.height:,} rows, {hw_e2.width} cols — OK")
print(f"E59: {hw_e59.height:,} rows, {hw_e59.width} cols — OK")

## Split temporal + winsorización

Aplica `split_temporal` y luego `winsorize_train_p99` (INV-1, INV-6).
Ambas funciones provienen de `src/evaluation/splits.py`.

In [ ]:

# Pipeline INV-1: split → winsorize → norm stats → z-score → max_N → windows → Dataset
results_e2 = {}
results_e59 = {}

def prepare_corridor(hw: pl.DataFrame, label: str) -> pl.DataFrame:
    df_split = split_temporal(hw)
    train_df  = df_split.filter(pl.col("split") == "train")
    df_winsor, threshold = winsorize_train_p99(train_df)
    # Re-attach the winsorized train to the full frame
    non_train = df_split.filter(pl.col("split") != "train")
    df_full = pl.concat([df_winsor, non_train])
    print(f"{label}: split counts = {df_split.group_by('split').agg(pl.len()).sort('split')}")
    print(f"{label}: winsorize threshold = {threshold:.4f} min")
    return df_full

df_e2  = prepare_corridor(hw_e2,  "E2")
df_e59 = prepare_corridor(hw_e59, "E59")

## Estadísticas de normalización (train only)

Computa media y desviación estándar de `delta_t_min` por `(empresaid, direction)`
usando SOLO filas de entrenamiento (INV-2, AC-NORM-1).

In [ ]:

def compute_stats_for(df: pl.DataFrame, label: str) -> "NormalizationStats":
    train_only = df.filter(pl.col("split") == "train")
    stats = compute_normalization_stats(train_only)
    print(f"\n{label} normalization stats:")
    for key in sorted(stats.means.keys()):
        print(f"  (empresa={key[0]}, dir={key[1]}): mean={stats.means[key]:.4f}, std={stats.stds[key]:.4f}")
    return stats

stats_e2  = compute_stats_for(df_e2,  "E2")
stats_e59 = compute_stats_for(df_e59, "E59")

## Aplicar z-score

Añade columna `delta_t_min_z` a todos los splits (train/val/test) usando
las estadísticas derivadas exclusivamente del tren (INV-2, DL-8 — sin clipping).

In [ ]:

df_e2  = apply_zscore(df_e2,  stats_e2)
df_e59 = apply_zscore(df_e59, stats_e59)

# Sanity: train z-score should have mean ≈ 0 and std ≈ 1 per direction
for label, df, stats in [("E2", df_e2, stats_e2), ("E59", df_e59, stats_e59)]:
    train_z = df.filter(pl.col("split") == "train")
    print(f"\n{label} train z-score sanity:")
    for (emp, dirn) in sorted(stats.means.keys()):
        subset = train_z.filter(
            (pl.col("empresaid") == emp) & (pl.col("direction") == dirn)
        )["delta_t_min_z"].drop_nulls()
        if subset.len() > 0:
            print(f"  (empresa={emp}, dir={dirn}): mean_z={subset.mean():.4f}, std_z={subset.std():.4f}")

## Features de contexto

Codificación cíclica de hora y día de semana + flag de día atípico (DL-2).
Fallback gracioso a `atypical_flag=0` si el CSV está ausente.

In [ ]:

# Try to locate atypical_days.csv (graceful fallback per DL-2)
atypical_path = None
if Path("/kaggle/input").exists():
    candidates = list(Path("/kaggle/input").rglob("atypical_days.csv"))
    if candidates:
        atypical_path = candidates[0]
if atypical_path is None:
    local_candidates = list(Path(".").rglob("atypical_days.csv"))
    if local_candidates:
        atypical_path = local_candidates[0]

atypical_dates = load_atypical_days(atypical_path)
print(f"Atypical days loaded: {len(atypical_dates)} dates (path={atypical_path})")

df_e2  = encode_context(df_e2,  atypical_dates=atypical_dates)
df_e59 = encode_context(df_e59, atypical_dates=atypical_dates)

print(f"E2 context columns:  {[c for c in df_e2.columns  if c in CONTEXT_FEATURE_NAMES]}")
print(f"E59 context columns: {[c for c in df_e59.columns if c in CONTEXT_FEATURE_NAMES]}")

## Cómputo de max_N

`max_N = train-p99(n_buses - 1)` por `(empresaid, direction)` (DL-5, AC-MAXN-1).
Las snapshots de val/test que excedan `max_N` serán truncadas en el Dataset.

In [ ]:

def compute_maxn_for(df: pl.DataFrame, label: str) -> dict:
    train_only = df.filter(pl.col("split") == "train")
    max_n = compute_max_N(train_only, quantile=0.99)
    print(f"\n{label} max_N per (empresa, direction):")
    for key in sorted(max_n.keys()):
        print(f"  (empresa={key[0]}, dir={key[1]}): max_N={max_n[key]}")

    # Truncation rate on val and test (R-MAXN-TRUNCATION guard, AC-MAXN-3)
    for split_name in ["val", "test"]:
        split_df = df.filter(pl.col("split") == split_name)
        if split_df.is_empty():
            continue
        # Count snapshots where n_buses - 1 > max_N
        n_total = split_df.height
        n_truncated = 0
        for (emp, dirn), cap in max_n.items():
            subset = split_df.filter(
                (pl.col("empresaid") == emp) & (pl.col("direction") == dirn)
            )
            if "n_buses" in subset.columns:
                n_truncated += subset.filter(pl.col("n_buses") - 1 > cap).height
        rate = n_truncated / n_total if n_total > 0 else 0.0
        print(f"  {label} {split_name} truncation rate: {rate:.4%} ({n_truncated}/{n_total})")
    return max_n

max_n_e2  = compute_maxn_for(df_e2,  "E2")
max_n_e59 = compute_maxn_for(df_e59, "E59")

## Índices de ventanas deslizantes

`make_window_index` per split per corredor (T_in=12, T_out=1, stride=1).
No materializa las ventanas — solo crea el índice de `(slot, start_idx)`.

In [ ]:

T_IN  = DEFAULT_T_IN   # 12
T_OUT = DEFAULT_T_OUT  # 1

window_idx_e2  = {}
window_idx_e59 = {}

for split_name in ["train", "val", "test"]:
    idx_e2  = make_window_index(df_e2.filter(pl.col("split")  == split_name), T_in=T_IN, T_out=T_OUT)
    idx_e59 = make_window_index(df_e59.filter(pl.col("split") == split_name), T_in=T_IN, T_out=T_OUT)
    window_idx_e2[split_name]  = idx_e2
    window_idx_e59[split_name] = idx_e59
    print(f"E2  {split_name:5s}: {len(idx_e2):,} windows")
    print(f"E59 {split_name:5s}: {len(idx_e59):,} windows")

## HeadwayDataset — smoke test

Instancia el dataset para el split de entrenamiento de E2 y verifica que
`__getitem__` retorna 5 tensores con las formas correctas (INV-4, AC-DS-1..4).

In [ ]:

import torch
from torch.utils.data import DataLoader

# Instantiate HeadwayDataset for E2 train split
ds_e2_train = HeadwayDataset(
    df=df_e2.filter(pl.col("split") == "train"),
    window_index=window_idx_e2["train"],
    max_N_by_direction=max_n_e2,
    T_in=T_IN,
    T_out=T_OUT,
)
print(f"E2 train dataset: {len(ds_e2_train):,} windows")

# Smoke test: __getitem__(0) must return 5 tensors with expected shapes
sample = ds_e2_train[0]
for key, tensor in sample.items():
    print(f"  {key}: shape={tuple(tensor.shape)}, dtype={tensor.dtype}")

# Verify shape invariants (INV-4)
assert sample["input"].shape[0] == T_IN,  f"input T_IN mismatch: {sample['input'].shape}"
assert sample["target"].shape[0] == T_OUT, f"target T_OUT mismatch: {sample['target'].shape}"
assert sample["context"].shape == (T_IN, 5), f"context shape mismatch: {sample['context'].shape}"
print("Shape invariants: OK")

## DataLoader — smoke test

Itera un batch del DataLoader de entrenamiento con `collate_fn` (AC-DS-5, AC-DS-6).

In [ ]:

loader_e2_train = DataLoader(
    ds_e2_train,
    batch_size=32,
    collate_fn=collate_fn,
    shuffle=False,
)

batch = next(iter(loader_e2_train))
print("One batch shapes:")
for key, tensor in batch.items():
    print(f"  {key}: {tuple(tensor.shape)}, dtype={tensor.dtype}")
print(f"Batch size: {batch['input'].shape[0]}")

## Estadísticas del dataset (dataset_stats.csv)

Escribe métricas por `(corridor, direction, split)` a `/kaggle/working/dataset_stats.csv`.
Columnas: corridor, direction, split, n_rows, n_windows, max_N,
          mean_delta_t_min, std_delta_t_min, truncation_rate.

In [ ]:

stats_rows = []

for label, df, stats, max_n, win_idx in [
    ("E2",  df_e2,  stats_e2,  max_n_e2,  window_idx_e2),
    ("E59", df_e59, stats_e59, max_n_e59, window_idx_e59),
]:
    for split_name in ["train", "val", "test"]:
        split_df = df.filter(pl.col("split") == split_name)
        n_rows   = split_df.height
        n_windows = len(win_idx[split_name])

        for (emp, dirn), cap in max_n.items():
            mean_val = stats.means.get((emp, dirn), float("nan"))
            std_val  = stats.stds.get((emp, dirn), float("nan"))

            # Truncation rate (0 for train by definition)
            if split_name == "train" or "n_buses" not in split_df.columns:
                trunc_rate = 0.0
            else:
                subset = split_df.filter(
                    (pl.col("empresaid") == emp) & (pl.col("direction") == dirn)
                )
                n_total_sub = subset.height
                n_trunc_sub = subset.filter(pl.col("n_buses") - 1 > cap).height if n_total_sub > 0 else 0
                trunc_rate  = n_trunc_sub / n_total_sub if n_total_sub > 0 else 0.0

            stats_rows.append({
                "corridor":         label,
                "direction":        dirn,
                "split":            split_name,
                "n_rows":           n_rows,
                "n_windows":        n_windows,
                "max_N":            cap,
                "mean_delta_t_min": mean_val,
                "std_delta_t_min":  std_val,
                "truncation_rate":  trunc_rate,
            })

stats_df = pl.DataFrame(stats_rows)
stats_df.write_csv(STATS_CSV)
print(f"CSV written to: {STATS_CSV}")
print(stats_df)